# Baseline Models

Before deploying complex forecasting models (ARIMA, VAR, machine learning, etc.),
it is essential to establish **baseline benchmarks**. A baseline is a simple model
that any serious forecasting method should be able to beat.

**Why baselines matter:**
- If your complex model can't beat a simple baseline, something is wrong.
- Baselines provide a lower bound for acceptable performance.
- They help you understand the inherent difficulty of the forecasting problem.

**Models covered:**
1. **Naive Forecast** (Random Walk)
2. **Seasonal Naive**
3. **Drift Method** (Random Walk with Drift)
4. **Simple Moving Average (SMA)**

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from forecastbox.metrics import mae, rmse, mape, mase
from forecastbox.cv import expanding_window_cv
from forecastbox.auto._baselines import NaiveBaseline, SeasonalNaiveBaseline, DriftBaseline

import sys
sys.path.insert(0, "..")
from utils.helpers import load_macro_brazil, load_macro_us, plot_series, plot_forecast

%matplotlib inline
plt.rcParams["figure.figsize"] = (12, 5)

In [ ]:
# Load data
df_brazil = load_macro_brazil()
gdp = df_brazil["gdp_growth"].dropna()
inflation = df_brazil["inflation"].dropna()

# Train/test split (80/20)
n = len(gdp)
split = int(0.8 * n)
gdp_train, gdp_test = gdp.iloc[:split], gdp.iloc[split:]
inf_train, inf_test = inflation.iloc[:split], inflation.iloc[split:]
h = len(gdp_test)

print(f"GDP growth: {n} total, {split} train, {n - split} test")
print(f"Inflation:  {len(inflation)} total, {split} train, {len(inflation) - split} test")
print(f"Forecast horizon: {h} months")

## 1. Naive Forecast (Random Walk)

The simplest possible forecast: predict that the next value equals the last observed value.

$$\hat{y}_{T+h} = y_T \quad \text{for all } h$$

This is equivalent to saying "tomorrow will be like today." Despite its simplicity,
the naive forecast is surprisingly hard to beat for many economic and financial series
(this is related to the efficient market hypothesis).

In [ ]:
# Naive forecast for GDP growth
naive = NaiveBaseline()
naive.fit(gdp_train)
naive_fc = naive.forecast(h)

print(f"Last training value: {gdp_train.iloc[-1]:.4f}")
print(f"Naive forecast (constant): {naive_fc.point[0]:.4f}")

# Evaluate
naive_mae = mae(gdp_test.values, naive_fc.point)
naive_rmse = rmse(gdp_test.values, naive_fc.point)
print(f"\nNaive MAE:  {naive_mae:.4f}")
print(f"Naive RMSE: {naive_rmse:.4f}")

# Plot
fig, ax = plt.subplots(figsize=(12, 5))
ax.plot(gdp_train.index, gdp_train.values, label="Training", color="#2196F3")
ax.plot(gdp_test.index, gdp_test.values, label="Actual (test)", color="#4CAF50")
ax.plot(gdp_test.index, naive_fc.point, label="Naive forecast", color="#F44336",
        linestyle="--", linewidth=2)
ax.axvline(gdp_test.index[0], color="gray", linestyle=":", alpha=0.7)
ax.set_title("Naive Forecast: GDP Growth", fontsize=14)
ax.set_ylabel("GDP Growth (%)")
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 2. Seasonal Naive

For data with seasonality, the seasonal naive forecast repeats the **last observed seasonal cycle**:

$$\hat{y}_{T+h} = y_{T+h-m} \quad \text{where } m \text{ is the seasonal period}$$

For monthly data with annual seasonality, $m = 12$: the forecast for January next year
equals the value from January this year.

This baseline is appropriate when the series exhibits clear seasonal patterns.

In [ ]:
# Seasonal naive for inflation (monthly data, period=12)
snaive = SeasonalNaiveBaseline(seasonal_period=12)
snaive.fit(inf_train)
snaive_fc = snaive.forecast(len(inf_test))

snaive_mae = mae(inf_test.values, snaive_fc.point)
snaive_rmse = rmse(inf_test.values, snaive_fc.point)
print(f"Seasonal Naive (m=12) for Inflation:")
print(f"  MAE:  {snaive_mae:.4f}")
print(f"  RMSE: {snaive_rmse:.4f}")

# Compare with simple naive
naive_inf = NaiveBaseline()
naive_inf.fit(inf_train)
naive_inf_fc = naive_inf.forecast(len(inf_test))
naive_inf_mae = mae(inf_test.values, naive_inf_fc.point)
print(f"\nSimple Naive for Inflation:")
print(f"  MAE:  {naive_inf_mae:.4f}")

# Plot
fig, ax = plt.subplots(figsize=(12, 5))
ax.plot(inf_train.index[-36:], inf_train.values[-36:], label="Training (last 3yr)",
        color="#2196F3")
ax.plot(inf_test.index, inf_test.values, label="Actual", color="#4CAF50")
ax.plot(inf_test.index, snaive_fc.point, label="Seasonal Naive", color="#F44336",
        linestyle="--", linewidth=2)
ax.plot(inf_test.index, naive_inf_fc.point, label="Simple Naive", color="#FF9800",
        linestyle=":", linewidth=2)
ax.axvline(inf_test.index[0], color="gray", linestyle=":", alpha=0.7)
ax.set_title("Seasonal Naive vs Simple Naive: Inflation", fontsize=14)
ax.set_ylabel("Inflation (%)")
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 3. Drift Method

The drift method is a random walk with a **linear trend** (drift):

$$\hat{y}_{T+h} = y_T + h \cdot \left(\frac{y_T - y_1}{T - 1}\right)$$

The drift term $\frac{y_T - y_1}{T - 1}$ is the average change per period.
This model extrapolates a straight line from the first to the last observation.

**Best for:** series with a clear upward or downward trend.

In [ ]:
# Drift forecast for GDP growth
drift = DriftBaseline()
drift.fit(gdp_train)
drift_fc = drift.forecast(h)

print(f"Drift estimate: {drift._drift:.6f} per month")
print(f"Last training value: {gdp_train.iloc[-1]:.4f}")
print(f"Drift forecast range: [{drift_fc.point[0]:.4f}, {drift_fc.point[-1]:.4f}]")

drift_mae = mae(gdp_test.values, drift_fc.point)
drift_rmse = rmse(gdp_test.values, drift_fc.point)
print(f"\nDrift MAE:  {drift_mae:.4f}")
print(f"Drift RMSE: {drift_rmse:.4f}")

# Plot: drift vs naive
fig, ax = plt.subplots(figsize=(12, 5))
ax.plot(gdp_train.index[-36:], gdp_train.values[-36:], label="Training (last 3yr)",
        color="#2196F3")
ax.plot(gdp_test.index, gdp_test.values, label="Actual", color="#4CAF50")
ax.plot(gdp_test.index, drift_fc.point, label="Drift", color="#9C27B0",
        linestyle="--", linewidth=2)
ax.plot(gdp_test.index, naive_fc.point, label="Naive", color="#F44336",
        linestyle=":", linewidth=2)
ax.axvline(gdp_test.index[0], color="gray", linestyle=":", alpha=0.7)
ax.set_title("Drift vs Naive: GDP Growth", fontsize=14)
ax.set_ylabel("GDP Growth (%)")
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 4. Simple Moving Average

The Simple Moving Average (SMA) forecasts the next value as the average of the last $k$ observations:

$$\hat{y}_{T+1} = \frac{1}{k}\sum_{i=0}^{k-1} y_{T-i}$$

**Key trade-off:**
- **Small $k$** (e.g., 3): responds quickly to changes, but noisy
- **Large $k$** (e.g., 12): smoother, but slow to adapt

SMA is a natural baseline that captures the recent level of a series without assuming
any trend or seasonality.

In [ ]:
# SMA with different window sizes
windows = [3, 6, 12]
sma_results = {}

fig, ax = plt.subplots(figsize=(12, 5))
ax.plot(gdp_test.index, gdp_test.values, label="Actual", color="#4CAF50", linewidth=2)

colors = ["#F44336", "#FF9800", "#9C27B0"]
for k, color in zip(windows, colors):
    # SMA forecast: use the mean of last k training values as a constant forecast
    sma_value = gdp_train.iloc[-k:].mean()
    sma_pred = np.full(h, sma_value)

    sma_mae = mae(gdp_test.values, sma_pred)
    sma_rmse = rmse(gdp_test.values, sma_pred)
    sma_results[k] = {"MAE": sma_mae, "RMSE": sma_rmse, "forecast": sma_value}

    ax.plot(gdp_test.index, sma_pred, label=f"SMA-{k} (MAE={sma_mae:.4f})",
            color=color, linestyle="--", linewidth=1.5)
    print(f"SMA-{k:2d}: forecast={sma_value:.4f}, MAE={sma_mae:.4f}, RMSE={sma_rmse:.4f}")

ax.axvline(gdp_test.index[0], color="gray", linestyle=":", alpha=0.7)
ax.set_title("Simple Moving Average: GDP Growth", fontsize=14)
ax.set_ylabel("GDP Growth (%)")
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 5. Comparing All Baselines

Now let's compare all baseline models using cross-validation to get a more robust
performance estimate. We use expanding window CV with an initial window of 60 months.

In [ ]:
# Define model functions for CV
def naive_fn(train: pd.Series) -> np.ndarray:
    model = NaiveBaseline()
    model.fit(train)
    return model.forecast(12).point

def snaive_fn(train: pd.Series) -> np.ndarray:
    model = SeasonalNaiveBaseline(seasonal_period=12)
    model.fit(train)
    return model.forecast(12).point

def drift_fn(train: pd.Series) -> np.ndarray:
    model = DriftBaseline()
    model.fit(train)
    return model.forecast(12).point

def sma3_fn(train: pd.Series) -> np.ndarray:
    return np.full(12, train.iloc[-3:].mean())

def sma6_fn(train: pd.Series) -> np.ndarray:
    return np.full(12, train.iloc[-6:].mean())

def sma12_fn(train: pd.Series) -> np.ndarray:
    return np.full(12, train.iloc[-12:].mean())

# Run CV for each model
models = {
    "Naive": naive_fn,
    "Seasonal Naive": snaive_fn,
    "Drift": drift_fn,
    "SMA-3": sma3_fn,
    "SMA-6": sma6_fn,
    "SMA-12": sma12_fn,
}

cv_results = {}
for name, fn in models.items():
    result = expanding_window_cv(
        data=gdp,
        model_fn=fn,
        initial_window=60,
        horizon=12,
        step=6,
    )
    cv_results[name] = result

# Build comparison table
rows = []
for name, res in cv_results.items():
    rows.append({
        "Model": name,
        "CV MAE": res.metrics_overall.get("mae", np.nan),
        "CV RMSE": res.metrics_overall.get("rmse", np.nan),
        "Folds": res.n_folds,
    })

comparison = pd.DataFrame(rows).set_index("Model").sort_values("CV MAE")
print("Baseline Comparison: GDP Growth (Expanding Window CV)")
print("=" * 55)
print(comparison.round(4))
print(f"\nBest model by MAE: {comparison.index[0]}")

In [ ]:
# Visual comparison
fig, ax = plt.subplots(figsize=(10, 6))

model_names = comparison.index.tolist()
mae_values = comparison["CV MAE"].values
rmse_values = comparison["CV RMSE"].values

x = np.arange(len(model_names))
width = 0.35

ax.barh(x - width/2, mae_values, width, label="MAE", color="#2196F3", alpha=0.8)
ax.barh(x + width/2, rmse_values, width, label="RMSE", color="#F44336", alpha=0.8)

ax.set_yticks(x)
ax.set_yticklabels(model_names)
ax.set_xlabel("Error")
ax.set_title("Baseline Model Comparison: GDP Growth", fontsize=14)
ax.legend()
ax.grid(True, alpha=0.3, axis="x")
plt.tight_layout()
plt.show()

## Exercise 1: Find the best baseline for unemployment

Compare all baseline models (naive, seasonal naive, drift, SMA-3, SMA-6, SMA-12)
for Brazil's unemployment series using expanding window cross-validation.
Which model performs best? Does it make sense given the characteristics of unemployment data?

In [ ]:
# TODO: Exercise 1 - Compare baselines for BR unemployment

## Exercise 2: Build a baseline for US CPI inflation

Load the US dataset (`macro_us.csv`) and build baseline forecasts for `cpi_inflation`.
Compare at least 3 baseline models and visualize the results.

In [ ]:
# TODO: Exercise 2 - Baselines for US CPI